# LSTM i GRU



## LSTM (*Long Short-Term Memory*)

L'LSTM va ser proposada per Hochreiter i Schmidhuber el 1997 com a solució al problema del vanishing gradient de les RNN simples. La idea clau és introduir un **cell state** $c_t$: un vector de memòria a llarg termini que flueix al llarg de la seqüència i que la xarxa pot aprendre a llegir, escriure i esborrar de manera selectiva mitjançant tres portes (*gates*):

- **Forget gate** $f_t$: decideix quina informació del cell state anterior s'oblida
- **Input gate** $i_t$: decideix quina informació nova s'escriu al cell state
- **Output gate** $o_t$: decideix quina part del cell state es fa servir per generar l'hidden state $h_t$

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(c_t)$$

on $\sigma$ és la funció sigmoide i $\odot$ és el producte element a element.


### Paràmetres de `nn.LSTM`

`nn.LSTM` comparteix els mateixos paràmetres que `nn.RNN` amb un afegit:

- `input_size`: nombre de variables d'entrada en cada pas temporal.
- `hidden_size`: dimensió del vector d'hidden state $h_t$ i del cell state $c_t$.
- `num_layers`: nombre de capes LSTM apilades.
- `batch_first`: si `True`, el tensor d'entrada s'espera amb forma `[batch, seq_len, features]`.
- `dropout`: aplica dropout entre capes quan `num_layers > 1`.
- `bidirectional`: si `True`, crea una LSTM bidireccional que processa la seqüència en els dos sentits.

> **Nota:** a diferència de `nn.RNN`, el mètode `forward` de `nn.LSTM` retorna `(out, (h_n, c_n))`, és a dir, tant l'hidden state com el cell state. Cal tenir-ho en compte al definir el `forward` del model.

### LSTMModel

```python
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(out[:, -1, :])  # agafem l'últim pas temporal
        return out.squeeze()
```

### GRU (*Gated Recurrent Unit*)

La GRU va ser proposada per Cho et al. el 2014 com una simplificació de l'LSTM. Elimina el cell state i redueix les tres portes a dues:

- **Reset gate** $r_t$: controla quanta informació de l'hidden state anterior es descarta
- **Update gate** $z_t$: controla quant de l'hidden state anterior es conserva i quant es substitueix per informació nova

$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)$$
$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)$$
$$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h)$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### Paràmetres de `nn.GRU`

`nn.GRU` té exactament els mateixos paràmetres que `nn.RNN`:

- `input_size`: nombre de variables d'entrada en cada pas temporal
- `hidden_size`: dimensió del vector d'hidden state $h_t$
- `num_layers`: nombre de capes GRU apilades
- `batch_first`: si `True`, el tensor d'entrada s'espera amb forma `[batch, seq_len, features]`
- `dropout`: aplica dropout entre capes quan `num_layers > 1`
- `bidirectional`: si `True`, crea una GRU bidireccional

> **Nota:** a diferència de l'LSTM, `nn.GRU` retorna `(out, h_n)`, sense cell state, igual que `nn.RNN`.

### GRUModel

```python
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, h_n = self.gru(x)
        out = self.fc(out[:, -1, :])  # agafem l'últim pas temporal
        return out.squeeze()
```


## Diferències entre RNN, LSTM i GRU

| | RNN | LSTM | GRU |
|---|---|---|---|
| Memòria a llarg termini | No | Sí (cell state) | Sí (update gate) |
| Nombre de portes | 0 | 3 | 2 |
| Paràmetres | Pocs | Molts | Intermedi |
| Velocitat d'entrenament | Ràpida | Lenta | Intermèdia |
| Vanishing gradient | Sí | No | No |

### Quan usar cada una

**RNN:** seqüències curtes on no cal memòria a llarg termini. En la pràctica, rarament és la millor opció.

**LSTM:** quan les dependències a llarg termini són importants i es disposa de prou dades i capacitat computacional. És l'opció per defecte en la majoria de problemes de sèries temporals.

**GRU:** quan es vol un bon equilibri entre rendiment i eficiència computacional. Amb datasets petits o resources limitats, sovint dona resultats similars a l'LSTM amb menys temps d'entrenament.

---

## Exercici: comparació RNN, LSTM i GRU

Usant el dataset **Air Passengers** i la mateixa preparació de dades de la secció anterior, entrena els tres models (RNN, LSTM i GRU) amb la mateixa arquitectura (`hidden_size=32`, `num_layers=1`, `epochs=200`, `lr=0.001`) i omple la taula de resultats:

| Model | MAE | RMSE | MAPE (%) |
|-------|-----|------|----------|
| RNN   |     |      |          |
| LSTM  |     |      |          |
| GRU   |     |      |          |

Visualitza les prediccions dels tres models en un mateix gràfic per facilitar la comparació.

> **Reflexió:** quin model ha funcionat millor? Els resultats s'expliquen per les diferències arquitecturals entre els tres models?
m